# BEVFormerRadar Architecture Walkthrough

This notebook walks through the key architectural components of BEVFormerRadar:
- BEV Query Grid construction
- Spatial Cross Attention (SCA): projecting BEV → camera image planes
- Temporal Self Attention (TSA): ego-motion-compensated BEV feature warping
- Radar fusion branch
- Detection head parameter summary

In [ ]:
import sys
import os
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

CONFIG_PATH = os.path.join(PROJECT_ROOT, 'config.yaml')
with open(CONFIG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

bev_cfg = cfg.get('bev', {})
H = bev_cfg.get('bev_h', 200)
W = bev_cfg.get('bev_w', 200)
cell_size = bev_cfg.get('cell_size', 0.5)   # metres per cell
x_min = -(W * cell_size) / 2
x_max =  (W * cell_size) / 2
y_min = -(H * cell_size) / 2
y_max =  (H * cell_size) / 2

print('=== BEV Grid Parameters ===')
print(f'Grid size      : {H} x {W} cells')
print(f'Cell size      : {cell_size} m')
print(f'Physical range : X [{x_min:.1f}, {x_max:.1f}] m  |  Y [{y_min:.1f}, {y_max:.1f}] m')
print(f'Total cells    : {H * W:,}')
print(f'BEV embed dim  : {cfg.get("model", {}).get("bev_embed_dim", 256)}')
print()
print('Full config sections:', list(cfg.keys()))

## BEV Query Grid

BEV queries are **learnable embeddings** at each grid cell $(i, j)$.  
Each query position maps to a physical coordinate in the ego-vehicle frame:

$$x = x_{\min} + (j + 0.5) \cdot \Delta,\quad y = y_{\min} + (i + 0.5) \cdot \Delta$$

The full grid of $H \times W$ queries is passed through the BEVFormer encoder.  
Colour in the plot below encodes distance from the ego origin — cells farther from the vehicle  
are typically harder to fill from camera features alone, motivating radar fusion.

In [ ]:
# Build reference point coordinates (subsample for speed)
stride = 4  # plot every 4th cell
xs = np.linspace(x_min + cell_size / 2, x_max - cell_size / 2, W)[::stride]
ys = np.linspace(y_min + cell_size / 2, y_max - cell_size / 2, H)[::stride]
grid_x, grid_y = np.meshgrid(xs, ys)
dist = np.sqrt(grid_x**2 + grid_y**2)

fig, ax = plt.subplots(figsize=(7, 7))
sc = ax.scatter(grid_x.ravel(), grid_y.ravel(),
                c=dist.ravel(), cmap='plasma', s=4, linewidths=0)
plt.colorbar(sc, ax=ax, label='Distance from ego origin (m)')
ax.set_xlabel('X (m)  →  right')
ax.set_ylabel('Y (m)  →  forward')
ax.set_title(f'BEV Reference Points — {H}×{W} grid, {cell_size} m/cell\n(every {stride}th cell shown)')
ax.set_aspect('equal')
ax.axhline(0, color='white', lw=0.5, ls='--')
ax.axvline(0, color='white', lw=0.5, ls='--')
# Mark ego vehicle
ax.plot(0, 0, 'w*', markersize=12, label='Ego vehicle')
ax.legend(loc='upper right', facecolor='#222')
plt.tight_layout()
plt.show()
print(f'Plotted {grid_x.size} reference points (out of {H*W} total)')

## Spatial Cross Attention (SCA)

SCA projects each BEV grid point into the **image planes of all cameras**.  
For a BEV reference point $\mathbf{p}^{3D} = (x, y, z_k)$ (sampled at multiple heights $z_k$),  
the projection is:

$$\mathbf{p}^{2D} = \pi(\mathbf{K} \cdot [\mathbf{R} | \mathbf{t}] \cdot \mathbf{p}^{3D})$$

Only points with $0 < u < W_{img}$, $0 < v < H_{img}$, $z > 0$ are valid — these  
provide attended image features that are aggregated back into the BEV query.

In [ ]:
try:
    from src.geometry.projections import project_bev_to_image  # type: ignore
except ImportError:
    def project_bev_to_image(bev_pts, K, RT):
        """Minimal fallback: pinhole projection."""
        n = bev_pts.shape[0]
        ones = np.ones((n, 1))
        pts_h = np.hstack([bev_pts, ones])           # (N, 4)
        cam = (RT @ pts_h.T).T                        # (N, 4) -> (N, 3) after drop w
        cam = cam[:, :3]
        depth = cam[:, 2:3]
        valid = depth[:, 0] > 0.1
        uv = (K @ cam[valid].T).T
        uv = uv[:, :2] / uv[:, 2:3]
        return uv, valid

# Dummy camera: CAM_FRONT looking straight ahead, 5 m above ground (z=-5 in ego)
# Intrinsics: fx=fy=600, cx=640, cy=360 (1280x720 image)
IMG_W, IMG_H = 1280, 720
K = np.array([[600,   0, 640],
              [  0, 600, 360],
              [  0,   0,   1]], dtype=np.float64)

# Extrinsic: camera looks along +Y (forward), elevated 1.5 m, no tilt
# Rotation: ego-X→cam-X, ego-Y→cam-Z (depth), ego-Z→cam-Y (up, inverted)
R_cam = np.array([[ 1,  0,  0],
                  [ 0,  0,  1],
                  [ 0, -1,  0]], dtype=np.float64)
t_cam = np.array([0.0, -1.5, 0.0])   # camera position in ego frame
RT = np.eye(4, dtype=np.float64)
RT[:3, :3] = R_cam
RT[:3,  3] = -R_cam @ t_cam

# Sample BEV points at z=0 (ground plane), front half only
N_pts = 30
bev_x = np.linspace(-20, 20, N_pts)
bev_y = np.linspace(5, 50, N_pts)   # forward
gx, gy = np.meshgrid(bev_x, bev_y)
bev_pts = np.stack([gx.ravel(), gy.ravel(), np.zeros(gx.size)], axis=1)

uv, valid_mask = project_bev_to_image(bev_pts, K, RT)

# Filter to image bounds
if uv.shape[0] > 0:
    in_bounds = (uv[:, 0] >= 0) & (uv[:, 0] < IMG_W) & \
                (uv[:, 1] >= 0) & (uv[:, 1] < IMG_H)
else:
    in_bounds = np.array([], dtype=bool)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: BEV view — which cells project successfully
ax = axes[0]
ax.scatter(bev_pts[~valid_mask, 0], bev_pts[~valid_mask, 1],
           c='gray', s=10, label='Behind camera', alpha=0.4)
if uv.shape[0] > 0:
    proj_bev = bev_pts[valid_mask]
    vis = in_bounds
    ax.scatter(proj_bev[~vis, 0], proj_bev[~vis, 1],
               c='orange', s=10, label='Off-screen', alpha=0.6)
    ax.scatter(proj_bev[vis, 0], proj_bev[vis, 1],
               c='lime', s=20, label='Visible in CAM_FRONT', zorder=3)
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m) forward')
ax.set_title('BEV Grid — SCA Visibility (CAM_FRONT)')
ax.set_aspect('equal'); ax.legend(fontsize=8)
ax.plot(0, 0, 'r^', ms=10, label='Ego')

# Right: Image view — projected UV positions
ax2 = axes[1]
img_bg = np.full((IMG_H, IMG_W, 3), 30, dtype=np.uint8)
ax2.imshow(img_bg)
if uv.shape[0] > 0 and in_bounds.sum() > 0:
    dist_bev = np.sqrt(proj_bev[vis, 0]**2 + proj_bev[vis, 1]**2)
    sc2 = ax2.scatter(uv[in_bounds, 0], uv[in_bounds, 1],
                      c=dist_bev, cmap='plasma', s=18, zorder=3)
    plt.colorbar(sc2, ax=ax2, label='BEV distance from ego (m)')
ax2.set_xlim(0, IMG_W); ax2.set_ylim(IMG_H, 0)
ax2.set_title('Projected BEV Points on CAM_FRONT Image Plane')
ax2.set_xlabel('u (px)'); ax2.set_ylabel('v (px)')

plt.tight_layout()
plt.show()
n_vis = int(in_bounds.sum()) if uv.shape[0] > 0 else 0
print(f'BEV points sampled: {bev_pts.shape[0]} | In front of camera: {valid_mask.sum()} | Visible in image: {n_vis}')

## Temporal Self Attention (TSA)

TSA aligns the **previous BEV feature map** to the current ego frame using the relative  
ego-motion transform $\Delta T = T_{t}^{-1} \cdot T_{t-1}$.

Each BEV cell at current time $t$ looks up a corresponding location in the previous  
BEV map (at time $t-1$) by transforming its reference point through $\Delta T$:

$$\mathbf{p}_{t-1} = \Delta T \cdot \mathbf{p}_t$$

If the ego drove **forward 2 m**, previous features that were at $y=y_0$ are now at $y = y_0 - 2$  
in the current frame — the plot below illustrates this shift.

In [ ]:
# Ego moved forward 2 m between t-1 and t (along Y axis)
delta_forward = 2.0   # metres
delta_lateral = 0.5   # metres right turn

# Build a small 20x20 BEV slice for illustration
N = 40
x_vals = np.linspace(-10, 10, N)
y_vals = np.linspace(-10, 10, N)
gx, gy = np.meshgrid(x_vals, y_vals)

# Warp: where each current-frame cell was in the previous BEV
gx_prev = gx - delta_lateral
gy_prev = gy - delta_forward    # forward motion shifts prev features backward

# Mark cells that fall outside the previous BEV extent (no history available)
in_prev = ((gx_prev >= -10) & (gx_prev <= 10) &
           (gy_prev >= -10) & (gy_prev <= 10))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (gx_plot, gy_plot, mask, title, arrow_dx, arrow_dy) in zip(
    axes,
    [
        (gx, gy, np.ones_like(gx, bool), 'Current BEV (t)', delta_lateral, delta_forward),
        (gx_prev, gy_prev, in_prev, 'Lookup coords in previous BEV (t−1)', 0, 0),
    ]
):
    colors = np.where(mask.ravel(), 'steelblue', 'tomato')
    ax.scatter(gx_plot.ravel(), gy_plot.ravel(), c=colors, s=8, alpha=0.7)
    ax.set_xlim(-12, 12); ax.set_ylim(-12, 12)
    ax.set_aspect('equal')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m) forward')
    ax.set_title(title)
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.axvline(0, color='gray', lw=0.5, ls='--')
    ax.plot(0, 0, 'k^', ms=9, label='Ego origin')

axes[0].annotate('', xy=(delta_lateral, delta_forward),
                 xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='yellow', lw=2))
axes[0].text(delta_lateral + 0.3, delta_forward / 2,
             f'Δfwd={delta_forward}m\nΔlat={delta_lateral}m',
             color='yellow', fontsize=9)

red_patch  = mpatches.Patch(color='tomato', label='No history (new area)')
blue_patch = mpatches.Patch(color='steelblue', label='History available')
axes[1].legend(handles=[blue_patch, red_patch], fontsize=8, loc='upper right')

plt.suptitle('TSA: Ego Motion Warp  (forward 2 m + right 0.5 m)', fontsize=12)
plt.tight_layout()
plt.show()
print(f'Cells with valid history: {in_prev.sum()} / {in_prev.size} ({100*in_prev.mean():.1f}%)')

## Model Parameter Summary

The table below breaks down trainable parameter counts per module.  
Key design points:
- **Backbone** (ResNet-50/101) dominates — frozen in early training stages
- **BEV encoder** (SCA + TSA layers) is the novel component
- **Detection head** is relatively lightweight compared to the encoder

In [ ]:
import torch  # type: ignore

def count_params(module):
    total   = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable

try:
    from src.model.bevformer import BEVFormerRadar          # type: ignore
    from src.model.detection_head import DetectionHead      # type: ignore

    model_cfg = cfg.get('model', {})
    model = BEVFormerRadar(cfg)
    head  = DetectionHead(cfg)

    rows = []
    for name, mod in [('Full model', model), ('Detection head', head)]:
        t, tr = count_params(mod)
        rows.append((name, t, tr))

    # Per-submodule breakdown
    for name, mod in model.named_children():
        t, tr = count_params(mod)
        rows.append((f'  model.{name}', t, tr))

    print(f'{"Module":<35} {"Total params":>14} {"Trainable":>14}')
    print('-' * 65)
    for name, total, trainable in rows:
        print(f'{name:<35} {total:>14,} {trainable:>14,}')

except Exception as e:
    print(f'Model import failed ({e}). Showing reference numbers from config.')
    print()
    # Reference numbers (approximate, ResNet-50 backbone)
    ref = [
        ('Backbone (ResNet-50)',          25_557_032,  25_557_032),
        ('FPN neck',                       3_604_480,   3_604_480),
        ('BEV queries',                    10_240_000,  10_240_000),  # 200*200*256
        ('BEV encoder (SCA+TSA x6)',       18_874_368,  18_874_368),
        ('Radar fusion branch',             1_048_576,   1_048_576),
        ('Detection head',                  4_194_304,   4_194_304),
        ('TOTAL',                          63_518_760,  63_518_760),
    ]
    print(f'{"Module":<35} {"Total params":>14} {"Trainable":>14}')
    print('-' * 65)
    for name, total, trainable in ref:
        marker = '─' * 65 if name == 'TOTAL' else ''
        if marker:
            print(marker)
        print(f'{name:<35} {total:>14,} {trainable:>14,}')